# Demand forecasting with Timestral: promotions and prices as covariates

*Author: Abdelaziz Guelfane ([@Abdelaziz03](https://github.com/Abdelaziz03)), CentraleSupélec, Université Paris-Saclay*

[Timestral](https://github.com/mistralai/timestral) is Mistral AI's family of time series foundation models. It forecasts a series it has never seen (zero-shot), returns quantiles instead of a single number, and runs on a CPU.

This cookbook forecasts daily product demand, the kind of series a retailer plans stock with. You will:

1. forecast 28 days ahead from the sales history alone,
2. read the quantile forecast and plot a prediction interval,
3. forecast through days with no sales record,
4. add what the planner already knows about the future (the promotion calendar and the prices) and what is only known about the past (store traffic),
5. score every variant against a seasonal-naive baseline.

The data is synthetic and generated in the notebook, so nothing has to be downloaded besides the model weights, and the true effect of a promotion is known.

## Setup

The package is installed from the Timestral repository, pinned to a commit so that the notebook stays reproducible. It needs Python 3.10 or later and PyTorch 2.4 or later, both available on Colab.

In [ ]:
!pip install -q "timestral @ git+https://github.com/mistralai/timestral.git@a534bdbd244d64f3d9d743f46c7bf6fc330ed4ff" "matplotlib>=3.8" "pandas>=2.0"

In [ ]:
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from timestral import TimestralForecaster

for module in (np, pd, matplotlib, torch):
    print(f"{module.__name__}=={module.__version__}")

## Load a model

Timestral comes in four sizes. The weights are downloaded once from the Hugging Face Hub.

| model | Hub repository | weights |
|---|---|---|
| Timestral-5M | `mistralai/Timestral-5M` | 21 MB |
| Timestral-28M | `mistralai/Timestral-28M` | 111 MB |
| Timestral-123M | `mistralai/Timestral-123M` | 490 MB |
| Timestral-209M | `mistralai/Timestral-209M` | 837 MB |

This notebook uses the 28M model. Change `MODEL_ID` to try another size.

In [ ]:
MODEL_ID = "mistralai/Timestral-28M"

forecaster = TimestralForecaster.from_pretrained(MODEL_ID)  # add device="cuda" to run on a GPU
print("patch size:", forecaster.patch_size, "time steps")
print("known-future covariates supported:", forecaster.supports_known_covariates)

## A synthetic retail dataset

Each product has two years of daily sales, built from pieces a demand planner would recognise:

- a base level and a slow trend,
- a weekly pattern with busier weekends, and a yearly pattern,
- **store traffic**, shared by all products. It drives sales, and it is only known for the past.
- **promotions**, one week long. A promotion lowers the price by 20% and lifts demand. The calendar is known in advance.
- **price**, which moves a little over time. Demand reacts to it. It is known in advance too.

Sales are drawn from a Poisson distribution around the expected demand. The last 28 days are held out to score the forecasts, and every product has one promotion inside those 28 days.

In [ ]:
HORIZON = 28  # days to forecast
NUM_DAYS = 2 * 365 + HORIZON
NUM_PRODUCTS = 8
PROMO_LIFT = 0.6  # a promotion multiplies demand by 1.6, on top of the price effect
PRICE_ELASTICITY = 1.5


def make_retail_demand(num_products, num_days, horizon, seed=0):
    rng = np.random.default_rng(seed)
    days = np.arange(num_days)
    weekly = 1.0 + 0.25 * np.sin(2 * np.pi * (days % 7) / 7) + 0.35 * (days % 7 >= 5)
    yearly = 1.0 + 0.2 * np.sin(2 * np.pi * days / 365.25)
    traffic = 1000 * weekly * yearly * rng.lognormal(0.0, 0.08, num_days)

    base = rng.uniform(20, 200, (num_products, 1))
    trend = 1.0 + rng.uniform(-0.2, 0.4, (num_products, 1)) * days / num_days

    promo = np.zeros((num_products, num_days))
    for product in range(num_products):
        start = rng.integers(0, 60)
        while start < num_days - horizon - 7:
            promo[product, start : start + 7] = 1.0
            start += rng.integers(40, 80)
        # One promotion inside the forecast window.
        start = num_days - horizon + rng.integers(0, horizon - 7)
        promo[product, start : start + 7] = 1.0

    list_price = rng.uniform(2, 30, (num_products, 1))
    drift = 1.0 + 0.05 * np.sin(2 * np.pi * days / 180 + rng.uniform(0, 6, (num_products, 1)))
    price = list_price * drift * (1.0 - 0.2 * promo)

    expected = (
        base
        * trend
        * (traffic / traffic.mean())
        * (1.0 + PROMO_LIFT * promo)
        * (price / list_price) ** (-PRICE_ELASTICITY)
    )
    demand = rng.poisson(expected)
    return (
        demand.astype(np.float32),
        promo.astype(np.float32),
        price.astype(np.float32),
        traffic.astype(np.float32),
    )


demand, promo, price, traffic = make_retail_demand(NUM_PRODUCTS, NUM_DAYS, HORIZON)
print("demand:", demand.shape, "| promo:", promo.shape, "| price:", price.shape, "| traffic:", traffic.shape)

In [ ]:
def shade_promotions(ax, promo_row, first_day):
    days = np.flatnonzero(promo_row) + first_day
    for day in days:
        ax.axvspan(day - 0.5, day + 0.5, color="tab:orange", alpha=0.15, linewidth=0)


product = 0
window = 180
first_day = NUM_DAYS - window
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(np.arange(first_day, NUM_DAYS), demand[product, -window:], color="black", linewidth=1)
shade_promotions(ax, promo[product, -window:], first_day)
ax.axvline(NUM_DAYS - HORIZON - 0.5, color="gray", linestyle="--")
ax.set_title(f"Product {product}: last {window} days (promotions shaded, held-out days right of the dashed line)")
ax.set_xlabel("day")
ax.set_ylabel("units sold")
plt.show()

## Zero-shot forecast

`forecast` takes the history as an array of shape `(series, variates, time)`. Here every product is its own series with one variate, so the history is `(products, 1, time)`.

The result holds one forecast per quantile level, in `forecast.quantiles` of shape `(series, variates, horizon, levels)`. `forecast.median` and `forecast.quantile(level)` select one level.

In [ ]:
context = demand[:, :-HORIZON]  # (products, time): what the model sees
actual = demand[:, -HORIZON:]  # (products, HORIZON): held out

forecast = forecaster.forecast(context[:, None, :], prediction_length=HORIZON)

print("quantiles:", tuple(forecast.quantiles.shape))
print("levels:", forecast.quantile_levels)

In [ ]:
def plot_forecast(forecast, product=0, title="", history_days=90):
    median = forecast.median[product, 0].cpu().numpy()
    low = forecast.quantile(0.1)[product, 0].cpu().numpy()
    high = forecast.quantile(0.9)[product, 0].cpu().numpy()

    last_context_day = context.shape[-1]
    history = np.arange(last_context_day - history_days, last_context_day)
    future = np.arange(last_context_day, last_context_day + HORIZON)

    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.plot(history, context[product, -history_days:], color="black", linewidth=1, label="history")
    ax.plot(future, actual[product], color="black", linewidth=1, linestyle=":", label="actual")
    ax.plot(future, median, color="tab:blue", label="median forecast")
    ax.fill_between(future, low, high, color="tab:blue", alpha=0.2, label="10% to 90% interval")
    shade_promotions(ax, promo[product, -(history_days + HORIZON) :], last_context_day - history_days)
    ax.set_title(title or f"Product {product}")
    ax.set_xlabel("day")
    ax.set_ylabel("units sold")
    ax.legend(loc="upper left", ncols=4, fontsize=8)
    plt.show()


plot_forecast(forecast, product=0, title="Product 0: history only")

The model only saw past sales here. It has no way to know that a promotion is scheduled inside the forecast window (the shaded days), so look at how the interval behaves on those days.

## Days without a record

Real sales histories have gaps: a stockout, a closed store, a broken export. Mark those days as `NaN` and pass the history as it is. The model treats `NaN` as missing instead of as zero sales.

In [ ]:
context_with_gaps = context.copy()
context_with_gaps[:, -60:-46] = np.nan  # two weeks without any record

forecast_gaps = forecaster.forecast(context_with_gaps[:, None, :], prediction_length=HORIZON)

print("missing days per product:", int(np.isnan(context_with_gaps[0]).sum()))
print("forecast is finite:", bool(torch.isfinite(forecast_gaps.quantiles).all()))
change = (forecast_gaps.median - forecast.median).abs().mean() / forecast.median.abs().mean()
print(f"mean change of the median forecast: {100 * float(change):.1f}%")

## Covariates

A covariate is a series that helps to forecast the target and is not forecast itself. Timestral takes two kinds:

- `known_covariates`, known over the history **and** the forecast window, of shape `(series, covariates, time + horizon)`. The promotion calendar and the price are known in advance. A covariate that only takes the values 0 and 1, like the promotion flag, is treated as an indicator.
- `past_covariates`, known over the history only, of shape `(series, covariates, time)`. Store traffic is measured, not planned.

Categorical covariates (a store type, a holiday name) go in `categorical_known_covariates` and `categorical_past_covariates` as numeric codes.

In [ ]:
known_covariates = np.stack([promo, price], axis=1)  # (products, 2, time + HORIZON)
past_covariates = np.broadcast_to(traffic[:-HORIZON], (NUM_PRODUCTS, 1, context.shape[-1])).copy()
print("known covariates:", known_covariates.shape, "| past covariates:", past_covariates.shape)

forecast_known = forecaster.forecast(
    context[:, None, :],
    prediction_length=HORIZON,
    known_covariates=known_covariates,
)
forecast_known_and_past = forecaster.forecast(
    context[:, None, :],
    prediction_length=HORIZON,
    known_covariates=known_covariates,
    past_covariates=past_covariates,
)

plot_forecast(forecast_known, product=0, title="Product 0: with the promotion calendar and the price")

## Forecasting related products together

So far every product was a separate series. Products of the same store share traffic, weather and seasonality, so they can also be passed as the **variates of one series**, of shape `(1, products, time)`. The model then forecasts them jointly and each product can use the history of the others.

In [ ]:
forecast_joint = forecaster.forecast(context[None, :, :], prediction_length=HORIZON)
print("quantiles:", tuple(forecast_joint.quantiles.shape))  # (1, products, HORIZON, levels)

## Scoring the forecasts

Three scores, all computed on the 28 held-out days:

- **MASE** compares the absolute error of the median forecast with the error of repeating the value from seven days earlier over the history. Below 1 means better than that in-sample seasonal-naive forecast.
- **WQL**, the weighted quantile loss, scores the whole set of quantiles. Lower is better.
- **Coverage** is the share of actual values inside the 10% to 90% interval. A calibrated interval covers about 80% of them.

The baseline repeats the last observed week. It has no quantiles, so it only gets a MASE.

In [ ]:
def seasonal_naive(history, horizon, season=7):
    repeats = int(np.ceil(horizon / season))
    return np.tile(history[:, -season:], repeats)[:, :horizon]


def mase(actual, prediction, history, season=7):
    scale = np.abs(history[:, season:] - history[:, :-season]).mean(axis=1, keepdims=True)
    return float((np.abs(actual - prediction) / scale).mean())


def weighted_quantile_loss(actual, quantiles, levels):
    levels = np.asarray(levels)
    error = actual[..., None] - quantiles
    pinball = np.maximum(levels * error, (levels - 1.0) * error)
    return float(2.0 * pinball.sum() / (len(levels) * np.abs(actual).sum()))


def coverage(actual, low, high):
    return float(((actual >= low) & (actual <= high)).mean())


def per_product(forecast):
    # Both layouts, (products, 1, ...) and (1, products, ...), become (products, HORIZON, levels).
    return forecast.quantiles.cpu().numpy().reshape(NUM_PRODUCTS, HORIZON, -1)


variants = {
    "history only": forecast,
    "history only, products forecast jointly": forecast_joint,
    "+ promotion calendar and price": forecast_known,
    "+ promotion calendar, price and past traffic": forecast_known_and_past,
}
levels = forecast.quantile_levels
on_promotion = promo[:, -HORIZON:] == 1

naive = seasonal_naive(context, HORIZON)
rows = [
    {
        "forecast": "seasonal naive (last week repeated)",
        "MASE": mase(actual, naive, context),
        "MAE on promotion days": float(np.abs(actual - naive)[on_promotion].mean()),
        "MAE on other days": float(np.abs(actual - naive)[~on_promotion].mean()),
    }
]
for name, variant in variants.items():
    quantiles = per_product(variant)
    median = quantiles[..., levels.index(0.5)]
    rows.append(
        {
            "forecast": f"{MODEL_ID.split('/')[-1]}, {name}",
            "MASE": mase(actual, median, context),
            "WQL": weighted_quantile_loss(actual, quantiles, levels),
            "coverage of the 10%-90% interval": coverage(
                actual, quantiles[..., levels.index(0.1)], quantiles[..., levels.index(0.9)]
            ),
            "MAE on promotion days": float(np.abs(actual - median)[on_promotion].mean()),
            "MAE on other days": float(np.abs(actual - median)[~on_promotion].mean()),
        }
    )

scores = pd.DataFrame(rows).set_index("forecast")
scores.round(3)

The last two columns split the absolute error of the median between the days with a promotion and the other days. That split is where a known promotion calendar should matter: without it, a forecast can only guess that a promotion might happen.

With eight products and one forecast window this is an illustration, not a benchmark. To compare variants on your own data, score several windows (a rolling backtest) and many series.

## How long does a forecast take?

The cell below times one call on this machine for the two smallest models, after a first call to warm up.

In [ ]:
for model_id in ("mistralai/Timestral-5M", "mistralai/Timestral-28M"):
    model = TimestralForecaster.from_pretrained(model_id)
    model.forecast(context[:, None, :], prediction_length=HORIZON)  # warm-up
    start = time.perf_counter()
    model.forecast(context[:, None, :], prediction_length=HORIZON)
    elapsed = time.perf_counter() - start
    print(f"{model_id}: {elapsed:.2f} s for {NUM_PRODUCTS} series of {context.shape[-1]} days")

## Recap

- Pass the history as `(series, variates, time)`. Separate series are forecast independently, the variates of one series jointly.
- The forecast is a set of quantiles. Use `forecast.median` for a point forecast and `forecast.quantile(level)` for an interval.
- Mark missing values as `NaN`.
- Put what you know about the future in `known_covariates` and what you only know about the past in `past_covariates`.
- Score against a simple baseline before trusting any forecast.

To go further, see the [Timestral repository](https://github.com/mistralai/timestral) and the model cards on the [Hugging Face Hub](https://huggingface.co/mistralai).